In [ ]:
import gc
import json
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedShuffleSplit

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
sns.set_theme(style="whitegrid")

# Διαδρομές αρχείων
train_path = Path("processed/train_final_scaled.csv")
test_path = Path("processed/test_final_scaled.csv")
scaler_path = Path("processed/minmax_scaler_final.joblib")

assert train_path.exists() and test_path.exists(), "Processed datasets not found!"

Path("models").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)

# Φόρτωση δεδομένων με float32 για εξοικονόμηση RAM
columns = pd.read_csv(train_path, nrows=0).columns.tolist()
features = [col for col in columns if col != "Label"]
dtypes = {col: "float32" for col in features}
dtypes["Label"] = "int16"

train_df = pd.read_csv(train_path, dtype=dtypes)
test_df = pd.read_csv(test_path, dtype=dtypes)

X_train = np.ascontiguousarray(train_df[features].to_numpy(dtype=np.float32))
X_test = np.ascontiguousarray(test_df[features].to_numpy(dtype=np.float32))

# Binary targets: 0 = Benign (Label == 1), 1 = Attack (Label != 1)
BENIGN_LABEL = 1
y_train_raw = train_df["Label"].to_numpy(dtype=np.int16)
y_test_raw = test_df["Label"].to_numpy(dtype=np.int16)
y_train = (y_train_raw != BENIGN_LABEL).astype(np.int64)
y_test = (y_test_raw != BENIGN_LABEL).astype(np.int64)

# Απελευθέρωση DataFrames από τη RAM
del train_df, test_df
gc.collect()

print("=== DATASET LOADED ===")
print(f"X_train: {X_train.shape} | Benign: {int((y_train == 0).sum()):,} | Attack: {int((y_train == 1).sum()):,}")
print(f"X_test:  {X_test.shape} | Benign: {int((y_test == 0).sum()):,} | Attack: {int((y_test == 1).sum()):,}")
print(f"Features: {len(features)} | SHAP version: {shap.__version__}")


In [ ]:
def evaluate_binary(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "FPR": float(fp / (fp + tn)),
        "ROC_AUC": float(roc_auc_score(y_true, y_prob)),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }

# Εκπαίδευση Baseline Random Forest (44 features)
t_start = time.perf_counter()
rf_base = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    n_jobs=-1,
)
rf_base.fit(X_train, y_train)
base_train_time = time.perf_counter() - t_start

# Αξιολόγηση στο Testing Set
base_probs = rf_base.predict_proba(X_test)[:, 1]
base_preds = (base_probs >= 0.5).astype(np.int64)
base_metrics = evaluate_binary(y_test, base_preds, base_probs)

# Αποθήκευση Baseline μοντέλου & μέτρηση μεγέθους
rf_base_path = Path("models/rf_binary_44.joblib")
joblib.dump(rf_base, rf_base_path)
base_size_kb = rf_base_path.stat().st_size / 1024

tree_nodes = sum(est.tree_.node_count for est in rf_base.estimators_)
mean_depth = float(np.mean([est.get_depth() for est in rf_base.estimators_]))
max_depth = int(np.max([est.get_depth() for est in rf_base.estimators_]))

print("=== BASELINE RANDOM FOREST (44 FEATURES) ===")
print(f"Training time: {base_train_time:.2f} s")
print(f"Accuracy:      {base_metrics['Accuracy']:.6f}")
print(f"Precision:     {base_metrics['Precision']:.6f}")
print(f"Recall:        {base_metrics['Recall']:.6f}")
print(f"F1-score:      {base_metrics['F1']:.6f}")
print(f"FPR:           {base_metrics['FPR']:.6f}")
print(f"ROC-AUC:       {base_metrics['ROC_AUC']:.6f}")
print(f"Confusion Matrix: TN={base_metrics['TN']}, FP={base_metrics['FP']}, FN={base_metrics['FN']}, TP={base_metrics['TP']}")
print(f"Model size:    {base_size_kb:.2f} KB ({base_size_kb/1024:.3f} MB)")
print(f"Tree stats:    total_nodes={tree_nodes:,}, mean_depth={mean_depth:.1f}, max_depth={max_depth}")


In [ ]:
def extract_attack_shap(shap_vals):
    """Εξάγει τον 2D πίνακα (N, F) τιμών SHAP για την κλάση Attack (class 1)."""
    if isinstance(shap_vals, list):
        return np.asarray(shap_vals[1], dtype=np.float64)
    arr = np.asarray(shap_vals, dtype=np.float64)
    if arr.ndim == 3 and arr.shape[-1] == 2:
        return arr[:, :, 1]
    return arr

# Στρωματοποιημένο δείγμα 5.000 ροών από το Training Set (χωρίς leakage από Test!)
SHAP_SAMPLE_SIZE = 5000
sss = StratifiedShuffleSplit(n_splits=1, test_size=SHAP_SAMPLE_SIZE, random_state=42)
_, shap_idx = next(sss.split(X_train, y_train))
X_shap = np.ascontiguousarray(X_train[shap_idx])
y_shap = y_train[shap_idx]

print(f"Computing exact TreeSHAP values for {SHAP_SAMPLE_SIZE:,} stratified training flows...")
t0 = time.perf_counter()
explainer_base = shap.TreeExplainer(rf_base)
raw_shap_vals = explainer_base.shap_values(X_shap, check_additivity=True)
shap_compute_time = time.perf_counter() - t0

sv_attack = extract_attack_shap(raw_shap_vals)
base_val = float(np.asarray(explainer_base.expected_value).ravel()[1])

# Έλεγχος μαθηματικής προσθετικότητας (Local Accuracy)
probs_shap = rf_base.predict_proba(X_shap)[:, 1]
max_additivity_err = float(np.max(np.abs((base_val + sv_attack.sum(axis=1)) - probs_shap)))
assert max_additivity_err < 1e-4, f"Additivity check failed: {max_additivity_err}"

print(f"Done in {shap_compute_time:.2f} s | Shape: {sv_attack.shape}")
print(f"Expected base value P(Attack): {base_val:.6f}")
print(f"Max additivity error |base + sum(SHAP) - P(Attack)|: {max_additivity_err:.2e} (OK)")

# Πίνακας παγκόσμιας σημαντικότητας (Global Feature Importance)
mean_abs_shap = np.mean(np.abs(sv_attack), axis=0)
std_abs_shap = np.std(np.abs(sv_attack), axis=0)
gini_imp = rf_base.feature_importances_

importance_df = pd.DataFrame({
    "Feature": features,
    "Feature_Index": np.arange(len(features)),
    "Mean_Abs_SHAP": mean_abs_shap,
    "Std_Abs_SHAP": std_abs_shap,
    "Gini_Importance": gini_imp,
}).sort_values("Mean_Abs_SHAP", ascending=False).reset_index(drop=True)

importance_df["Rank"] = np.arange(1, len(importance_df) + 1)
importance_df["SHAP_Share_Pct"] = 100.0 * importance_df["Mean_Abs_SHAP"] / importance_df["Mean_Abs_SHAP"].sum()
importance_df["Cumulative_SHAP_Pct"] = importance_df["SHAP_Share_Pct"].cumsum()

cols_order = [
    "Rank", "Feature", "Feature_Index", "Mean_Abs_SHAP",
    "SHAP_Share_Pct", "Cumulative_SHAP_Pct", "Gini_Importance", "Std_Abs_SHAP"
]
importance_df = importance_df[cols_order]
importance_df.to_csv("results/q5_shap_feature_importance.csv", index=False)

# Συσχέτιση κατάταξης SHAP vs Gini
rho, pval = spearmanr(importance_df["Mean_Abs_SHAP"], importance_df["Gini_Importance"])
print(f"\nSpearman rank correlation (SHAP vs Gini): rho = {rho:.4f} (p = {pval:.2e})")

print("\n=== TOP 20 MOST IMPORTANT FEATURES (BY MEAN |SHAP|) ===")
print(importance_df.head(20)[["Rank", "Feature", "Mean_Abs_SHAP", "SHAP_Share_Pct", "Cumulative_SHAP_Pct", "Gini_Importance"]].round(6).to_string(index=False))

print("\n=== BOTTOM 10 LEAST IMPORTANT FEATURES ===")
print(importance_df.tail(10)[["Rank", "Feature", "Mean_Abs_SHAP", "SHAP_Share_Pct", "Cumulative_SHAP_Pct"]].round(6).to_string(index=False))


In [ ]:
# --- Plot 1: Global Feature Importance Bar Chart + Cumulative SHAP Curve ---
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

top20 = importance_df.head(20).iloc[::-1]
axes[0].barh(top20["Feature"], top20["Mean_Abs_SHAP"], color="#1f77b4", edgecolor="none")
axes[0].set_xlabel("Mean |SHAP value| (επίδραση σε P(Attack))", fontsize=11)
axes[0].set_title("Top 20 Σημαντικότερα Χαρακτηριστικά (TreeSHAP)", fontsize=12, fontweight="bold")
axes[0].grid(axis="x", linestyle="--", alpha=0.5)

ranks = importance_df["Rank"].to_numpy()
cum_pct = importance_df["Cumulative_SHAP_Pct"].to_numpy()
axes[1].plot(ranks, cum_pct, marker="o", markersize=4, linewidth=2, color="#d62728")
for target_pct, col in [(80, "#2ca02c"), (90, "#ff7f0e"), (95, "#9467bd")]:
    k_hit = int(ranks[np.argmax(cum_pct >= target_pct)])
    axes[1].axhline(target_pct, color=col, linestyle="--", alpha=0.6, linewidth=1.2)
    axes[1].axvline(k_hit, color=col, linestyle=":", alpha=0.7, linewidth=1.2)
    axes[1].annotate(
        f"{target_pct}% σε {k_hit} features",
        xy=(k_hit, target_pct),
        xytext=(k_hit + 2.5, target_pct - 6),
        fontsize=9.5,
        fontweight="bold",
        color=col,
        arrowprops=dict(arrowstyle="->", color=col, lw=1.0),
    )

axes[1].set_xlabel("Πλήθος Κορυφαίων Χαρακτηριστικών (k)", fontsize=11)
axes[1].set_ylabel("Αθροιστικό Ποσοστό Συνολικής |SHAP| Σημαντικότητας (%)", fontsize=11)
axes[1].set_title("Συγκέντρωση Επεξηγηματικής Ισχύος (Cumulative SHAP)", fontsize=12, fontweight="bold")
axes[1].set_xlim(1, 44)
axes[1].set_ylim(0, 103)
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("results/q5_shap_global_importance.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Plot 2: SHAP Summary Beeswarm Plot (Top 20 features) ---
X_shap_df = pd.DataFrame(X_shap, columns=features)
plt.figure(figsize=(11, 7.5))
shap.summary_plot(sv_attack, X_shap_df, max_display=20, show=False)
plt.title("SHAP Summary (Beeswarm) — Top 20 Features για Ανίχνευση Επιθέσεων", fontsize=12, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig("results/q5_shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Plot 3: Local Waterfall Explanations σε αθέατες ροές του Testing Set ---
scaler_bundle = joblib.load(scaler_path)
scaler = scaler_bundle["scaler"]

# Επιλογή ενός σαφούς True Positive (Attack) και ενός True Negative (Benign) από το X_test
tp_candidates = np.where((y_test == 1) & (base_probs >= 0.99))[0]
tn_candidates = np.where((y_test == 0) & (base_probs <= 0.01))[0]
tp_idx = int(tp_candidates[0])
tn_idx = int(tn_candidates[0])

for tag, idx, title_gr, fname in [
    ("ATTACK (TP)", tp_idx, "Τοπική Εξήγηση SHAP — Επίθεση (True Positive)", "results/q5_shap_local_attack.png"),
    ("BENIGN (TN)", tn_idx, "Τοπική Εξήγηση SHAP — Κανονική Κίνηση (True Negative)", "results/q5_shap_local_benign.png"),
]:
    x_row_scaled = X_test[idx:idx+1]
    x_row_raw = np.round(scaler.inverse_transform(x_row_scaled).ravel(), 2)
    sv_row = extract_attack_shap(explainer_base.shap_values(x_row_scaled, check_additivity=True)).ravel()
    prob_row = float(base_probs[idx])

    print(f"\n--- Local Example: {tag} (test_idx={idx}, original Label={int(y_test_raw[idx])}, P(Attack)={prob_row:.4f}) ---")
    top_local = np.argsort(np.abs(sv_row))[::-1][:6]
    for f_i in top_local:
        print(f"  {features[f_i]:22s} | raw={x_row_raw[f_i]:12.2f} | SHAP={sv_row[f_i]:+.5f}")

    exp_obj = shap.Explanation(
        values=sv_row,
        base_values=base_val,
        data=x_row_raw,
        feature_names=features,
    )
    plt.figure(figsize=(10, 5.5))
    shap.plots.waterfall(exp_obj, max_display=12, show=False)
    plt.title(f"{title_gr} | P(Attack)={prob_row:.4f}", fontsize=11, fontweight="bold", pad=12)
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
# Απελευθέρωση του baseline explainer πριν το loop
del explainer_base
gc.collect()

ranked_indices = importance_df["Feature_Index"].to_numpy(dtype=int)
ranked_features = importance_df["Feature"].tolist()

k_schedule = [44, 35, 30, 25, 20, 15, 12, 10, 8, 6, 5, 4, 3, 2, 1]

BATCH_SIZE = 512
N_SINGLE_INF = 2000
N_BATCH_INF_REPS = 5
N_SINGLE_XAI = 200
N_BATCH_XAI_REPS = 3

pruning_records = []
tmp_raw_path = Path("models/_tmp_rf_raw.joblib")
tmp_cmp_path = Path("models/_tmp_rf_cmp.joblib")

print(
    f"{'k':>3s} | {'Train(s)':>8s} | {'F1':>8s} | {'Recall':>8s} | {'FPR(%)':>8s} | "
    f"{'Size(KB)':>8s} | {'InfMed(ms)':>10s} | {'XAIMed(ms)':>10s} | {'Tot(ms)':>8s} | {'InfThr(f/s)':>11s} | {'XAIThr(f/s)':>11s}"
)
print("-" * 124)

for k in k_schedule:
    sel_idx = ranked_indices[:k]
    X_tr_k = np.ascontiguousarray(X_train[:, sel_idx])
    X_te_k = np.ascontiguousarray(X_test[:, sel_idx])

    # 1. Εκπαίδευση (Training Time)
    t0 = time.perf_counter()
    rf_k = RandomForestClassifier(
        n_estimators=50,
        random_state=42,
        n_jobs=-1,
    )
    rf_k.fit(X_tr_k, y_train)
    train_time_s = time.perf_counter() - t0

    # 2. Αξιολόγηση Classification Performance στο πλήρες Testing Set
    probs_k = rf_k.predict_proba(X_te_k)[:, 1]
    preds_k = (probs_k >= 0.5).astype(np.int64)
    m = evaluate_binary(y_test, preds_k, probs_k)

    # 3. Μέγεθος Μοντέλου & Δομική Πολυπλοκότητα
    joblib.dump(rf_k, tmp_raw_path)
    size_raw_kb = tmp_raw_path.stat().st_size / 1024
    joblib.dump(rf_k, tmp_cmp_path, compress=3)
    size_cmp_kb = tmp_cmp_path.stat().st_size / 1024

    total_nodes_k = int(sum(est.tree_.node_count for est in rf_k.estimators_))
    mean_depth_k = float(np.mean([est.get_depth() for est in rf_k.estimators_]))

    # 4. Inference Latency & Batch Throughput (n_jobs=1 όπως στο Ερώτημα 3)
    rf_k.set_params(n_jobs=1)

    for _ in range(100):
        rf_k.predict(X_te_k[:1])
    for _ in range(20):
        rf_k.predict(X_te_k[:BATCH_SIZE])

    inf_single_ms = []
    for i in range(N_SINGLE_INF):
        ts = time.perf_counter_ns()
        rf_k.predict(X_te_k[i:i+1])
        inf_single_ms.append((time.perf_counter_ns() - ts) / 1_000_000)

    inf_mean_ms = float(np.mean(inf_single_ms))
    inf_med_ms = float(np.median(inf_single_ms))
    inf_p95_ms = float(np.percentile(inf_single_ms, 95))

    batch_times = []
    for _ in range(N_BATCH_INF_REPS):
        ts = time.perf_counter()
        for i in range(0, len(X_te_k), BATCH_SIZE):
            rf_k.predict(X_te_k[i:i+BATCH_SIZE])
        batch_times.append(time.perf_counter() - ts)

    inf_batch_total_s = float(np.median(batch_times))
    inf_batch_thr = float(len(X_te_k) / inf_batch_total_s)
    inf_amort_ms = float(inf_batch_total_s / len(X_te_k) * 1000.0)

    # 5. XAI Latency κατά τη διάρκεια του Inference (TreeExplainer)
    explainer_k = shap.TreeExplainer(rf_k)

    # Warm-up XAI
    for _ in range(10):
        explainer_k.shap_values(X_te_k[:1], check_additivity=False)
    explainer_k.shap_values(X_te_k[:BATCH_SIZE], check_additivity=False)

    # Single-flow XAI latency
    xai_single_ms = []
    for i in range(N_SINGLE_XAI):
        ts = time.perf_counter_ns()
        explainer_k.shap_values(X_te_k[i:i+1], check_additivity=False)
        xai_single_ms.append((time.perf_counter_ns() - ts) / 1_000_000)

    xai_mean_ms = float(np.mean(xai_single_ms))
    xai_med_ms = float(np.median(xai_single_ms))
    xai_p95_ms = float(np.percentile(xai_single_ms, 95))

    # Batch XAI latency (512 flows)
    xai_batch_times = []
    for _ in range(N_BATCH_XAI_REPS):
        ts = time.perf_counter()
        explainer_k.shap_values(X_te_k[:BATCH_SIZE], check_additivity=False)
        xai_batch_times.append(time.perf_counter() - ts)

    xai_batch_med_s = float(np.median(xai_batch_times))
    xai_batch_thr = float(BATCH_SIZE / xai_batch_med_s)
    xai_amort_ms = float(xai_batch_med_s / BATCH_SIZE * 1000.0)

    total_single_med_ms = inf_med_ms + xai_med_ms
    xai_overhead_pct = (xai_med_ms / inf_med_ms) * 100.0
    xai_slowdown_x = total_single_med_ms / inf_med_ms
    batch_combined_thr = 1000.0 / (inf_amort_ms + xai_amort_ms)

    rec = {
        "Num_Features": k,
        "Features_Removed": 44 - k,
        "Feature_Reduction_Pct": round(100.0 * (44 - k) / 44.0, 2),
        "Train_Time_s": train_time_s,
        "Accuracy": m["Accuracy"],
        "Precision": m["Precision"],
        "Recall": m["Recall"],
        "F1": m["F1"],
        "FPR": m["FPR"],
        "ROC_AUC": m["ROC_AUC"],
        "TN": m["TN"],
        "FP": m["FP"],
        "FN": m["FN"],
        "TP": m["TP"],
        "Model_Size_KB": size_raw_kb,
        "Model_Size_Compressed_KB": size_cmp_kb,
        "Total_Tree_Nodes": total_nodes_k,
        "Mean_Tree_Depth": mean_depth_k,
        "Inf_Mean_ms": inf_mean_ms,
        "Inf_Median_ms": inf_med_ms,
        "Inf_P95_ms": inf_p95_ms,
        "Inf_Batch_Total_s": inf_batch_total_s,
        "Inf_Batch_Throughput_fps": inf_batch_thr,
        "Inf_Amortized_ms": inf_amort_ms,
        "XAI_Mean_ms": xai_mean_ms,
        "XAI_Median_ms": xai_med_ms,
        "XAI_P95_ms": xai_p95_ms,
        "XAI_Batch_512_ms": xai_batch_med_s * 1000.0,
        "XAI_Batch_Throughput_fps": xai_batch_thr,
        "XAI_Amortized_ms": xai_amort_ms,
        "Total_Single_Median_ms": total_single_med_ms,
        "XAI_Overhead_Pct": xai_overhead_pct,
        "XAI_Slowdown_Factor": xai_slowdown_x,
        "Combined_Batch_Throughput_fps": batch_combined_thr,
    }
    pruning_records.append(rec)

    print(
        f"{k:3d} | {train_time_s:8.2f} | {m['F1']:8.6f} | {m['Recall']:8.6f} | {m['FPR']*100:8.5f} | "
        f"{size_raw_kb:8.1f} | {inf_med_ms:10.4f} | {xai_med_ms:10.4f} | {total_single_med_ms:8.4f} | "
        f"{inf_batch_thr:11.1f} | {xai_batch_thr:11.1f}",
        flush=True
    )

    del rf_k, explainer_k, X_tr_k, X_te_k
    gc.collect()

# Καθαρισμός προσωρινών αρχείων
tmp_raw_path.unlink(missing_ok=True)
tmp_cmp_path.unlink(missing_ok=True)

pruning_df = pd.DataFrame(pruning_records)
pruning_df.to_csv("results/q5_feature_pruning_results.csv", index=False)
print("\nSaved: results/q5_feature_pruning_results.csv")


In [ ]:
# --- Plot 4: 2x2 Comprehensive Feature Pruning Dashboard ---
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
ks = pruning_df["Num_Features"].to_numpy()

# (a) Classification Performance vs k
ax = axes[0, 0]
ax.plot(ks, pruning_df["F1"], marker="o", linewidth=2, label="F1-score", color="#1f77b4")
ax.plot(ks, pruning_df["Recall"], marker="s", linewidth=1.8, linestyle="--", label="Recall", color="#2ca02c")
ax.plot(ks, pruning_df["Precision"], marker="^", linewidth=1.8, linestyle=":", label="Precision", color="#9467bd")
ax.set_xlabel("Πλήθος Χαρακτηριστικών (k)")
ax.set_ylabel("Μετρικές Ταξινόμησης")
ax.set_title("(α) Απόδοση Ταξινόμησης & FPR vs Πλήθος Features", fontweight="bold")
min_m = float(min(pruning_df["F1"].min(), pruning_df["Recall"].min(), pruning_df["Precision"].min()))
ax.set_ylim(max(0.0, min_m - 0.03), 1.015)
ax.invert_xaxis()
ax.grid(True, linestyle="--", alpha=0.5)

ax2 = ax.twinx()
ax2.plot(ks, pruning_df["FPR"] * 100.0, marker="d", linewidth=1.8, color="#d62728", label="FPR (%)")
ax2.set_ylabel("False Positive Rate (%)", color="#d62728")
ax2.tick_params(axis="y", labelcolor="#d62728")

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="center left", fontsize=9, frameon=True)

# (b) Training Time & Model Size vs k
ax = axes[0, 1]
l1 = ax.plot(ks, pruning_df["Train_Time_s"], marker="o", linewidth=2, color="#ff7f0e", label="Χρόνος Εκπαίδευσης (s)")
ax.set_xlabel("Πλήθος Χαρακτηριστικών (k)")
ax.set_ylabel("Χρόνος Εκπαίδευσης (s)", color="#ff7f0e")
ax.tick_params(axis="y", labelcolor="#ff7f0e")
ax.set_title("(β) Χρόνος Εκπαίδευσης & Μέγεθος Μοντέλου vs Features", fontweight="bold")
ax.invert_xaxis()
ax.grid(True, linestyle="--", alpha=0.5)

ax2 = ax.twinx()
l2 = ax2.plot(ks, pruning_df["Model_Size_KB"], marker="s", linewidth=2, color="#17becf", label="Μέγεθος Μοντέλου (KB)")
ax2.set_ylabel("Μέγεθος Μοντέλου — Uncompressed (KB)", color="#17becf")
ax2.tick_params(axis="y", labelcolor="#17becf")

lines = l1 + l2
ax.legend(lines, [l.get_label() for l in lines], loc="upper right", fontsize=9)

# (c) Single-flow Inference Latency vs XAI Latency vs Total
ax = axes[1, 0]
ax.plot(ks, pruning_df["Inf_Median_ms"], marker="o", linewidth=2, color="#1f77b4", label="Inference μόνο (Median ms)")
ax.plot(ks, pruning_df["XAI_Median_ms"], marker="s", linewidth=2, color="#d62728", label="SHAP TreeExplainer (Median ms)")
ax.plot(ks, pruning_df["Total_Single_Median_ms"], marker="^", linewidth=2.2, linestyle="--", color="#222222", label="Συνολικό: Inference + XAI (ms)")
ax.set_xlabel("Πλήθος Χαρακτηριστικών (k)")
ax.set_ylabel("Single-Flow Latency (ms)")
ax.set_title("(γ) Επίδραση Pruning σε Inference & XAI Latency", fontweight="bold")
ax.invert_xaxis()
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(loc="upper right", fontsize=9)

# (d) Batch Throughput: Inference-only vs Inference+XAI (log scale)
ax = axes[1, 1]
ax.plot(ks, pruning_df["Inf_Batch_Throughput_fps"], marker="o", linewidth=2, color="#2ca02c", label="Inference Batch Throughput (flows/s)")
ax.plot(ks, pruning_df["Combined_Batch_Throughput_fps"], marker="s", linewidth=2, color="#d62728", label="Inference + Inline XAI Throughput (flows/s)")
ax.set_yscale("log")
ax.set_xlabel("Πλήθος Χαρακτηριστικών (k)")
ax.set_ylabel("Throughput (flows/sec — log scale)")
ax.set_title("(δ) Μέγιστο Batch Throughput: Χωρίς vs Με Inline XAI", fontweight="bold")
ax.invert_xaxis()
ax.grid(True, which="both", linestyle="--", alpha=0.45)
ax.legend(loc="center right", fontsize=9)

plt.suptitle("Επίδραση Σταδιακής Αφαίρεσης Χαρακτηριστικών (SHAP Pruning) στο Random Forest IDS", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("results/q5_pruning_performance_and_cost.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Plot 5: Stacked Single-Flow Latency & Key Operating Points ---
key_ks = [44, 20, 15, 10, 5, 3]
sub = pruning_df[pruning_df["Num_Features"].isin(key_ks)].copy()
sub = sub.sort_values("Num_Features", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
x_pos = np.arange(len(sub))
x_labels = [f"{int(k)} feat." for k in sub["Num_Features"]]

# Left: Stacked latency breakdown
axes[0].bar(x_pos, sub["Inf_Median_ms"], width=0.55, label="Inference Latency (ms)", color="#1f77b4")
axes[0].bar(x_pos, sub["XAI_Median_ms"], width=0.55, bottom=sub["Inf_Median_ms"], label="SHAP XAI Latency (ms)", color="#d62728", alpha=0.85)
for i, (_, row) in enumerate(sub.iterrows()):
    axes[0].text(
        i, row["Total_Single_Median_ms"] + 0.15,
        f"{row['Total_Single_Median_ms']:.2f} ms\n({row['XAI_Slowdown_Factor']:.2f}×)",
        ha="center", va="bottom", fontsize=8.5, fontweight="bold"
    )
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(x_labels)
axes[0].set_ylabel("Median Single-Flow Latency (ms)")
axes[0].set_title("Ανάλυση Single-Flow Latency: Inference + XAI Overhead", fontweight="bold")
axes[0].set_ylim(0, sub["Total_Single_Median_ms"].max() * 1.22)
axes[0].legend(loc="upper right")
axes[0].grid(axis="y", linestyle="--", alpha=0.5)

# Right: Relative savings (% vs 44-feature baseline) for k=15, 10, 5
base_row = pruning_df[pruning_df["Num_Features"] == 44].iloc[0]
cmp_ks = [15, 10, 5]
cmp_sub = pruning_df[pruning_df["Num_Features"].isin(cmp_ks)].sort_values("Num_Features", ascending=False)

metrics_save = [
    ("Features", lambda r: 100.0 * (44 - r["Num_Features"]) / 44.0),
    ("Train Time", lambda r: 100.0 * (base_row["Train_Time_s"] - r["Train_Time_s"]) / base_row["Train_Time_s"]),
    ("Model Size", lambda r: 100.0 * (base_row["Model_Size_KB"] - r["Model_Size_KB"]) / base_row["Model_Size_KB"]),
    ("XAI Latency", lambda r: 100.0 * (base_row["XAI_Median_ms"] - r["XAI_Median_ms"]) / base_row["XAI_Median_ms"]),
]
bar_w = 0.24
m_pos = np.arange(len(metrics_save))
colors = ["#2ca02c", "#1f77b4", "#ff7f0e"]

all_vals = []
for j, (_, row) in enumerate(cmp_sub.iterrows()):
    vals = [fn(row) for _, fn in metrics_save]
    all_vals.extend(vals)
    bars = axes[1].bar(m_pos + (j - 1) * bar_w, vals, width=bar_w, label=f"Top-{int(row['Num_Features'])} (F1={row['F1']:.4f})", color=colors[j])
    for b, v in zip(bars, vals):
        lbl = f"-{v:.0f}%" if v >= 0 else f"+{abs(v):.0f}%"
        y_txt = v + 1.5 if v >= 0 else 1.5
        axes[1].text(b.get_x() + b.get_width()/2, y_txt, lbl, ha="center", va="bottom", fontsize=8, fontweight="bold")

axes[1].set_xticks(m_pos)
axes[1].set_xticklabels([m[0] for m in metrics_save])
axes[1].set_ylabel("Μείωση σε σχέση με Baseline 44 Features (%)")
axes[1].set_title("Κέρδη από Feature Pruning σε Επιλεγμένα Σημεία Λειτουργίας", fontweight="bold")
axes[1].set_ylim(min(0, min(all_vals) - 8), max(112, max(all_vals) + 22))
axes[1].legend(loc="upper center", ncol=1, fontsize=9, frameon=True)
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("results/q5_xai_latency_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Εύρεση βέλτιστου Pareto σημείου (ελάχιστο k με F1 >= 0.9990 και FPR <= 0.0001)
eligible = pruning_df[(pruning_df["F1"] >= 0.9990) & (pruning_df["FPR"] <= 0.0001)]
best_row = eligible.sort_values("Num_Features").iloc[0]
best_k = int(best_row["Num_Features"])
best_features = ranked_features[:best_k]

# Αποθήκευση βέλτιστου pruned μοντέλου
rf_best = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf_best.fit(np.ascontiguousarray(X_train[:, ranked_indices[:best_k]]), y_train)
best_model_path = Path(f"models/rf_binary_top{best_k}.joblib")
joblib.dump(rf_best, best_model_path)

# Σύγκριση Baseline (44) vs Optimal Pruned (best_k) vs Ultra-Compact (5)
base_row = pruning_df[pruning_df["Num_Features"] == 44].iloc[0]
row_15 = pruning_df[pruning_df["Num_Features"] == 15].iloc[0]
row_5 = pruning_df[pruning_df["Num_Features"] == 5].iloc[0]

summary_table = pruning_df[pruning_df["Num_Features"].isin(sorted({44, 15, best_k, 5}, reverse=True))][[
    "Num_Features", "Feature_Reduction_Pct", "Accuracy", "Precision", "Recall", "F1", "FPR",
    "Train_Time_s", "Model_Size_KB", "Inf_Median_ms", "XAI_Median_ms", "Total_Single_Median_ms",
    "Inf_Batch_Throughput_fps", "Combined_Batch_Throughput_fps"
]].copy()

print("=== KEY OPERATING POINTS COMPARISON ===")
print(summary_table.round(6).to_string(index=False))

def pct_change(new_val, old_val):
    return float(100.0 * (new_val - old_val) / old_val)

summary_json = {
    "model": "RandomForestClassifier (n_estimators=50, random_state=42)",
    "xai_method": "SHAP TreeExplainer (tree_path_dependent, exact Shapley values)",
    "shap_sample_size": SHAP_SAMPLE_SIZE,
    "shap_sample_source": "Stratified sample from Training Set (zero test leakage)",
    "max_additivity_error": max_additivity_err,
    "top_10_features": ranked_features[:10],
    "optimal_pareto_criterion": "Smallest k with F1 >= 0.9990 and FPR <= 0.0001 (0.01%)",
    "optimal_k": best_k,
    "optimal_features": best_features,
    "optimal_model_path": str(best_model_path),
    "baseline_44": {k: (float(v) if isinstance(v, (np.floating, float)) else int(v) if isinstance(v, (np.integer, int)) else v) for k, v in base_row.to_dict().items()},
    "optimal_pruned": {k: (float(v) if isinstance(v, (np.floating, float)) else int(v) if isinstance(v, (np.integer, int)) else v) for k, v in best_row.to_dict().items()},
    "deltas_optimal_vs_44_pct": {
        "features": pct_change(best_row["Num_Features"], base_row["Num_Features"]),
        "train_time": pct_change(best_row["Train_Time_s"], base_row["Train_Time_s"]),
        "model_size_kb": pct_change(best_row["Model_Size_KB"], base_row["Model_Size_KB"]),
        "inf_median_ms": pct_change(best_row["Inf_Median_ms"], base_row["Inf_Median_ms"]),
        "xai_median_ms": pct_change(best_row["XAI_Median_ms"], base_row["XAI_Median_ms"]),
        "total_single_ms": pct_change(best_row["Total_Single_Median_ms"], base_row["Total_Single_Median_ms"]),
        "inf_batch_throughput": pct_change(best_row["Inf_Batch_Throughput_fps"], base_row["Inf_Batch_Throughput_fps"]),
        "xai_batch_throughput": pct_change(best_row["XAI_Batch_Throughput_fps"], base_row["XAI_Batch_Throughput_fps"]),
        "f1_diff_abs": float(best_row["F1"] - base_row["F1"]),
    },
}

with open("results/q5_xai_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=2, ensure_ascii=False)

print(f"\nSaved optimal pruned model: {best_model_path} ({best_model_path.stat().st_size/1024:.2f} KB)")
print("Saved summary JSON:         results/q5_xai_summary.json")
print(f"\nOptimal Top-{best_k} Features: {best_features}")
